In [17]:
!pip install tensorflow

## Import necessary libraries


In [21]:

import os
import pandas as pd
import numpy as np
import json
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from transformers import BertTokenizer, BertModel, AdamW, BertTokenizerFast, BertForTokenClassification, AutoTokenizer
from sklearn.preprocessing import LabelEncoder

from tqdm import tqdm

## Installing necessary libraries
import tensorflow as tf
import pandas as pd
import numpy as np


import matplotlib.pyplot as plt
import seaborn as sns

import re
import nltk
import spacy
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

import gensim
from gensim.models import Word2Vec, Doc2Vec
from transformers import BertTokenizer, BertModel
import torch
from gensim.models.doc2vec import TaggedDocument
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint


from tensorflow.keras import layers, models
from tensorflow.keras.layers import Bidirectional
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, TimeDistributed, Bidirectional


import time
import unicodedata



In [22]:
!python -m spacy download en_core_web_md

nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()
nlp = spacy.load('en_core_web_md')
pd.set_option('display.max_colwidth',100)


tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
bert_model = BertModel.from_pretrained("bert-base-uncased")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.5/33.5 MB 8.1 MB/s eta 0:00:0000:0100:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_md')


[nltk_data] Downloading package punkt to /Users/monilshah/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/monilshah/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/monilshah/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


## Getting the data

In [23]:
training_data_path = '/Users/monilshah/Documents/02_NWU/10_MSDS_453_NLP/98_project_work/02_wip_data/relations_train_data.json'
test_data_path = '/Users/monilshah/Documents/02_NWU/10_MSDS_453_NLP/98_project_work/01_Input/02_wip_data/relations_test_data.json'

In [24]:
# Load the JSON file
with open(training_data_path, 'r') as file:
    all_data = json.load(file)

# Example structure of each data entry
# {
#     "news_line": "Sentence text here.",
#     "triples": [
#         {"subject": "Entity1", "object": "Entity2", "relation": "RelationType"}
#     ]
# }



In [25]:
data = all_data[0]

In [26]:
data[0]

{'news_line': 'NEW YORK (Reuters) - Apple Inc Chief Executive Steve Jobs sought to soothe investor concerns about his health on Monday, saying his weight loss was caused by a hormone imbalance that is relatively simple to treat.',
 'triples': [{'subject': 'Apple Inc',
   'object': 'Steve Jobs',
   'relation': 'founded_by'},
  {'subject': 'Apple Inc',
   'object': 'Steve Jobs',
   'relation': 'chief_executive_officer'}]}

In [27]:
# Expand each triple into a row
rows = []
for item in data:  # Loop through each item in the list
    for triple in item['triples']:
        rows.append({
            'news_line': item['news_line'],
            'subject': triple['subject'],
            'object': triple['object'],
            'relation': triple['relation']
        })

# Create the DataFrame
df = pd.DataFrame(rows)
print(df)


                                                                                                news_line  \
0     NEW YORK (Reuters) - Apple Inc Chief Executive Steve Jobs sought to soothe investor concerns abo...   
1     NEW YORK (Reuters) - Apple Inc Chief Executive Steve Jobs sought to soothe investor concerns abo...   
2     Last week, Citigroup Inc's ( C.N ) Chief Executive Vikram Pandit said that he, Chairman Win Bisc...   
3     Lehman Brothers LEH.N shares fell sharply on Monday on speculation that the investment bank coul...   
4     Lehman Brothers LEH.N shares fell sharply on Monday on speculation that the investment bank coul...   
...                                                                                                   ...   
8074  In particular, he said leases of used A330-200 aircraft from Airbus Group SE (Xetra: A1XBMK - ne...   
8075  The company is an omnipresent in households the world over thanks to its operations across multi...   
8076               

In [28]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

df['relation'] = le.fit_transform(df['relation'])

In [29]:
def get_filtered_entities(text):
    
    text = text.replace("Inc ", "Inc. ")
    
    doc = nlp(text)
    # Collect only entities that are either a person or an organization
    entities = set([ent.text for ent in doc.ents if ent.label_ in {"PERSON", "ORG"}])
    return entities

# Apply the function and convert the set to a comma-separated string
df['entities'] = df['news_line'].apply(lambda x: ', '.join(get_filtered_entities(x)))

print(df)


                                                                                                news_line  \
0     NEW YORK (Reuters) - Apple Inc Chief Executive Steve Jobs sought to soothe investor concerns abo...   
1     NEW YORK (Reuters) - Apple Inc Chief Executive Steve Jobs sought to soothe investor concerns abo...   
2     Last week, Citigroup Inc's ( C.N ) Chief Executive Vikram Pandit said that he, Chairman Win Bisc...   
3     Lehman Brothers LEH.N shares fell sharply on Monday on speculation that the investment bank coul...   
4     Lehman Brothers LEH.N shares fell sharply on Monday on speculation that the investment bank coul...   
...                                                                                                   ...   
8074  In particular, he said leases of used A330-200 aircraft from Airbus Group SE (Xetra: A1XBMK - ne...   
8075  The company is an omnipresent in households the world over thanks to its operations across multi...   
8076               

In [30]:


# Step 1: Tokenize sentences and create labels for each token
def prepare_data(df, maxlen = None):
    tokenizer = Tokenizer()
    tokenizer.fit_on_texts(df['news_line'])
    
    X = []
    y = []
    for _, row in df.iterrows():
        tokens = row['news_line'].split()
        labels = [1 if " ".join(tokens[i:i+len(row['object'].split())]) == row['object'] else 0 for i in range(len(tokens))]
        
        
        # Handle multi-word subjects by labeling each word in the subject with 1
        for i in range(len(tokens) - len(row['object'].split()) + 1):
            if " ".join(tokens[i:i + len(row['object'].split())]) == row['object']:
                labels[i:i + len(row['object'].split())] = [1] * len(row['object'].split())

        # Append tokenized text and labels
        X.append(tokens)
        y.append(labels)
    
    # Convert texts to sequences of integers
    X_seq = tokenizer.texts_to_sequences([" ".join(tokens) for tokens in X])
    
    if maxlen is None:
        maxlen = max(len(seq) for seq in X_seq)
    
    # Pad sequences and labels
    X_pad = pad_sequences(X_seq, padding='post', maxlen= maxlen)
    y_pad = pad_sequences(y, padding='post', maxlen= maxlen)

    return X_pad, np.array(y_pad), tokenizer



In [31]:
# Step 2: Split data into train, validation, and test sets
# First, split out the test data (20% test)
df_train_val, df_test = train_test_split(df, test_size=0.2, random_state=42)

# Now, split the remaining data into train and validation sets (80% train, 20% validation)
X, y, tokenizer = prepare_data(df_train_val)
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)




In [32]:
X.shape, y.shape

((6463, 998), (6463, 998))

In [33]:
# Step 3: Define LSTM model for sequence tagging
model = Sequential([
    Embedding(input_dim=len(tokenizer.word_index) + 1, output_dim=64, input_length=X.shape[1]),
    Bidirectional(LSTM(64, return_sequences=True)),
    TimeDistributed(Dense(1, activation='sigmoid'))
])

# Compile the model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Step 4: Train the model
model.fit(X_train, y_train, epochs=5, batch_size=10, validation_data=(X_val, y_val))

# Step 5: Evaluate the model on test data
X_test, y_test, _ = prepare_data(df_test)  # Prepare the test data
loss, accuracy = model.evaluate(X_test, y_test)
print(f"Test Accuracy: {accuracy:.2f}")

Epoch 1/5


/opt/anaconda3/envs/Conda_3_12_7/lib/python3.12/site-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


517/517 ━━━━━━━━━━━━━━━━━━━━ 249s 455ms/step - accuracy: 0.9984 - loss: 0.0243 - val_accuracy: 0.9986 - val_loss: 0.0055
Epoch 2/5
517/517 ━━━━━━━━━━━━━━━━━━━━ 286s 554ms/step - accuracy: 0.9986 - loss: 0.0048 - val_accuracy: 0.9986 - val_loss: 0.0043
Epoch 3/5
517/517 ━━━━━━━━━━━━━━━━━━━━ 302s 583ms/step - accuracy: 0.9987 - loss: 0.0037 - val_accuracy: 0.9987 - val_loss: 0.0040
Epoch 4/5
517/517 ━━━━━━━━━━━━━━━━━━━━ 1425s 3s/step - accuracy: 0.9989 - loss: 0.0030 - val_accuracy: 0.9987 - val_loss: 0.0040
Epoch 5/5
517/517 ━━━━━━━━━━━━━━━━━━━━ 1120s 2s/step - accuracy: 0.9990 - loss: 0.0026 - val_accuracy: 0.9987 - val_loss: 0.0042
51/51 ━━━━━━━━━━━━━━━━━━━━ 3s 63ms/step - accuracy: 0.9983 - loss: 0.0097
Test Accuracy: 1.00
